In [ ]:
!pip install --upgrade -q google-cloud-aiplatform datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.9 MB/s eta 0:00:00


In [ ]:
# ==========================================
# 2. KIMLIK DOGRULAMA VE PROJE AYARLARI
# ==========================================
import time
import json
import re
from datasets import load_dataset
from google.colab import auth
from google.colab import drive
import vertexai
from vertexai.generative_models import GenerativeModel


# Bu satır çalıştığında bir pencere açılacak, Google hesabını seçip onay ver.
auth.authenticate_user()

# Google Drive bağlama
drive.mount('/content/drive')

PROJECT_ID = "project-68c4b860-effa-4af9-897" # Senin resimdeki Proje ID'n
REGION = "us-central1"

vertexai.init(project=PROJECT_ID, location=REGION)

# En zeki modelimizi seçiyoruz
MODEL_NAME = "gemini-2.5-pro"
model = GenerativeModel(MODEL_NAME)

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [ ]:
# ==========================================
# 3. VERI SETI VE PARAMETRELER
# ==========================================

dataset = load_dataset("malhajar/gsm8k-tr", "main", split="train")
dataset = dataset.select(range(0, 2000)) # İlk aşamada 10 soru ile test ediyoruz

def build_prompt(question):
    return f"""
Aşağıdaki matematik sorusunu, alışılmış çözümün dışında farklı bir yaklaşımla çözmeye çalış.

Mümkünse farklı bir düşünme yolu kullan ve ardından sonucu bul.

Soru:
{question}

Sonucu cevabının en sonunda şu formatta ver:
#### <sayı>
"""

def extract_answer(text):
    if text is None:
        return None

    # Modelin üretebileceği markdown kalın yazı (**) veya gizli boşlukları temizle
    cleaned_text = text.replace("**", "").replace("\\", "").strip()

    # Senin orijinal regex'inin her ihtimale karşı (virgül dahil) geliştirilmiş hali
    match = re.search(r"####\s*(-?\d+[\.,]?\d*)", cleaned_text)
    if match:
        # Yakalanan değerdeki virgülü noktaya çevir (eğer ondalıklıysa Python float okuyabilsin diye)
        return match.group(1).replace(",", ".")

    return None

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/6.84k [00:00<?, ?B/s]

train-7472.jsonl:   0%|          | 0.00/5.46M [00:00<?, ?B/s]

test-1318.jsonl:   0%|          | 0.00/987k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7472 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1318 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# 0. OUTPUT DOSYASI VE RESUME
# ==========================================
output_file = "/content/drive/MyDrive/gsm8k_gemini_2_5_sonuclari-kritik-nokta.json"

try:
    with open(output_file, "r", encoding="utf-8") as f:
        results = json.load(f)
    print(f"✅ BAŞARILI: {len(results)} kayıt Drive'dan yüklendi. {len(results)}. indexten devam ediliyor...")
except Exception as e:
    print(f"❌ HATA: Dosya yüklenemedi! Hata sebebi: {e}")
    print("⚠️ UYARI: Eğer devam ederseniz eski verilerin üzerine yazılabilir veya 0'dan başlar!")
    results = []

# kaldığın yerden devam etmek için index
start_index = len(results)

✅ BAŞARILI: 1000 kayıt Drive'dan yüklendi. 1000. indexten devam ediliyor...


In [ ]:
# ==========================================
# 4. CALISTIRMA DÖNGÜSÜ (CHECKPOINT'LI)
# ==========================================
checkpoint_interval = 20

print(f"Başlatılıyor... Model: {MODEL_NAME}")

for i in range(start_index, len(dataset)):
    item = dataset[i]
    q_id = i

    question = item["question"]
    ground_truth = item["answer"]
    prompt = build_prompt(question)

    success = False
    retry_count = 0

    while not success and retry_count < 3:
        try:
            response = model.generate_content(
                prompt,
                generation_config={
                    "max_output_tokens": 4096,
                    "temperature": 0.1,
                }
            )

            output = response.text.strip()
            predicted = extract_answer(output)

            results.append({
                "id": q_id,
                "question": question,
                "ground_truth": ground_truth,
                "model_output": output,
                "predicted": predicted
            })

            print(f"[{q_id}] Tamamlandı: {predicted}")
            success = True

        except Exception as e:
            retry_count += 1
            if "429" in str(e):
                print(f"Limit uyarısı! 30 saniye mola... (Deneme {retry_count})")
                time.sleep(30)
            else:
                print(f"Hata: {e}")
                time.sleep(5)

    # 🔥 CHECKPOINT (HER 20 SORUDA)
    if (i + 1) % checkpoint_interval == 0:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f"Checkpoint kaydedildi ({i+1})")

    time.sleep(2)

Başlatılıyor... Model: gemini-2.5-pro
[1000] Tamamlandı: 1145
[1001] Tamamlandı: 113
[1002] Tamamlandı: 480
[1003] Tamamlandı: 1260
[1004] Tamamlandı: 40
[1005] Tamamlandı: 28
[1006] Tamamlandı: 18
[1007] Tamamlandı: 4
[1008] Tamamlandı: 25
[1009] Tamamlandı: 15
[1010] Tamamlandı: 8
[1011] Tamamlandı: 15620
[1012] Tamamlandı: 2
[1013] Tamamlandı: 6
[1014] Tamamlandı: 4
[1015] Tamamlandı: 52
[1016] Tamamlandı: 31
[1017] Tamamlandı: 4
[1018] Tamamlandı: 29
[1019] Tamamlandı: 15
Checkpoint kaydedildi (1020)
[1020] Tamamlandı: 30
[1021] Tamamlandı: 19
[1022] Tamamlandı: 800
[1023] Tamamlandı: 400
[1024] Tamamlandı: 640
[1025] Tamamlandı: 5
[1026] Tamamlandı: 240
[1027] Tamamlandı: 5100
[1028] Tamamlandı: 77
[1029] Tamamlandı: 84
[1030] Tamamlandı: 480
[1031] Tamamlandı: 8
[1032] Tamamlandı: 2400
[1033] Tamamlandı: 8
[1034] Tamamlandı: 960
[1035] Tamamlandı: 40
[1036] Tamamlandı: 1400
[1037] Tamamlandı: 9
[1038] Tamamlandı: 30
[1039] Tamamlandı: 33
Checkpoint kaydedildi (1040)
[1040] Tamaml

In [ ]:
# ==========================================
# 6. BİLDİRİM GÖNDER
# ==========================================

import requests
import json

def send_push_alert(title, message):
    # Uygulamanın sana verdiği API Key'i buraya yapıştır
    api_key = "3193d93e-6601-4ea7-995f-f693a6a1139d"

    # Dökümantasyondaki doğru URL yapısı:
    url = f"https://push.techulus.com/api/v1/notify/{api_key}"

    payload = {
        "title": title,
        "body": message
    }

    headers = {
        "Content-Type": "application/json"
    }

    try:
        # POST isteğini gönderiyoruz
        response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=10)

        if response.status_code == 200:
            print("Bildirim başarıyla gönderildi! ✅")
        else:
            print(f"Hata Kodu: {response.status_code}")
            print(f"Sunucu Yanıtı: {response.text}")

    except Exception as e:
        print(f"Bir bağlantı hatası oluştu: {e}")

# Test etmek için çalıştır:
send_push_alert("kardeş eğitim bitti", "git bilgisayarı kontrol et.")